# Correlation Analysis of Exercises

Install matplotlib

In [ ]:
%pip install matplotlib

Definisco le variabili necessarie (nella reale esecuzione queste variabili vengono importate da un file di configurazione esterno).

In [ ]:
import os

CLICKHOUSE_URL = os.getenv("CLICKHOUSE_JDBC_URL", "jdbc:clickhouse://clickhouse:8123/bigintensive")
CLICKHOUSE_PROPS = {
    "user": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", ""),
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}
CLICKHOUSE_TABLE = os.getenv("CLICKHOUSE_TABLE", "running_samples")


print("Variabili di ambiente importare")

Importo tutte le librerie necessarie e configuro l'ambiente Spark per eseguire l'analisi dei dati.

In [ ]:
import logging
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from itertools import combinations

print("Configurazione completata")

Rimuovo eventuali executor pod rimasti orfani da esecuzioni precedenti fallite, prima di creare una nuova sessione.

In [ ]:
import json
import ssl
import urllib.request

SA_DIR = "/var/run/secrets/kubernetes.io/serviceaccount"
NAMESPACE = "bigintensive"


def _k8s_request(path, method="GET"):
    with open(f"{SA_DIR}/token") as f:
        token = f.read()
    ctx = ssl.create_default_context(cafile=f"{SA_DIR}/ca.crt")
    req = urllib.request.Request(
        f"https://kubernetes.default{path}",
        method=method,
        headers={"Authorization": f"Bearer {token}"},
    )
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.load(resp)


def pulisci_executor_orfani():
    pods = _k8s_request(
        f"/api/v1/namespaces/{NAMESPACE}/pods?labelSelector=spark-role%3Dexecutor"
    )["items"]
    attivi = 0
    for pod in pods:
        nome = pod["metadata"]["name"]
        fase = pod["status"]["phase"]
        if fase in ("Failed", "Succeeded"):
            _k8s_request(f"/api/v1/namespaces/{NAMESPACE}/pods/{nome}", method="DELETE")
            print(f"eliminato pod {nome} ({fase})")
        else:
            attivi += 1

    # Le conf-map degli executor non vengono rimosse se il driver muore: bloccano il riavvio con 409.
    if attivi == 0:
        cms = _k8s_request(f"/api/v1/namespaces/{NAMESPACE}/configmaps")["items"]
        for cm in cms:
            nome = cm["metadata"]["name"]
            if nome.startswith("spark-exec-") and nome.endswith("-conf-map"):
                _k8s_request(
                    f"/api/v1/namespaces/{NAMESPACE}/configmaps/{nome}", method="DELETE"
                )
                print(f"eliminata configmap {nome}")
    else:
        print(f"{attivi} executor ancora attivi: configmap non toccate")

    print("pulizia completata")


pulisci_executor_orfani()
print("Executor orfani eliminati")

Creazione di una sessione Spark con le configurazioni appropriate per l'analisi dei dati dei runner.

In [ ]:
# In client mode la JVM del driver parte prima del builder: la memoria va fissata qui.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 2g pyspark-shell"

# Senza questo, gli executor non hanno ownerReference e restano orfani se il driver muore.
DRIVER_POD_NAME = os.environ["SPARK_DRIVER_POD_NAME"]

# Numero di executor: cambialo e riavvia il kernel per confrontare i tempi di esecuzione.
NUM_EXECUTORS = 4

# Partizioni shuffle fisse per un confronto corretto tra configurazioni di executor.
SHUFFLE_PARTITIONS = 16

# Un getOrCreate fallito lascia un SparkContext zombie che va chiuso prima di riprovare.
from pyspark import SparkContext

if SparkContext._active_spark_context is not None:
    try:
        SparkContext._active_spark_context.stop()
    except Exception as errore:
        print(f"contesto precedente non chiudibile: {errore}")

spark = (
    SparkSession.builder.appName("exercise-correlation-analysis")
    .config("spark.log.level", "ERROR")
    .config("spark.master", "k8s://https://kubernetes.default:443")
    .config("spark.kubernetes.namespace", "bigintensive")
    .config("spark.kubernetes.authenticate.driver.mounted", "true")
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark")
    .config("spark.kubernetes.authenticate.executor.mounted", "true")
    .config("spark.kubernetes.driver.pod.name", DRIVER_POD_NAME)
    .config("spark.kubernetes.executor.deleteOnTermination", "true")
    .config("spark.driver.host", "jupyter.bigintensive.svc.cluster.local")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.driver.blockManager.port", "7079")
    .config("spark.kubernetes.container.image", "apache/spark:3.5.3")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "2g")
    .config("spark.kubernetes.executor.request.cores", "400m")
    .config("spark.kubernetes.executor.limit.cores", "1")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.executor.instances", str(NUM_EXECUTORS))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.3,"
        "com.clickhouse:clickhouse-jdbc:0.6.3,"
        "com.clickhouse:clickhouse-http-client:0.6.3,"
        "org.apache.httpcomponents.client5:httpclient5:5.3.1",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"SparkSession creata con {NUM_EXECUTORS} executor richiesti")

Libraries import

In [ ]:
# Register the JDBC driver explicitly in the driver JVM.
spark._jvm.java.lang.Class.forName("com.clickhouse.jdbc.ClickHouseDriver")
spark.sparkContext.setLogLevel("WARN")

bounds_query = """
    (SELECT
        coalesce(min(athlete_id), 0) AS min_athlete_id,
        coalesce(max(athlete_id), 0) AS max_athlete_id
    FROM allenamenti) AS athlete_bounds
    """

bounds = (
    spark.read.format("jdbc")
    .option("url", CLICKHOUSE_URL)
    .option("dbtable", bounds_query)
    .option("user", CLICKHOUSE_PROPS["user"])
    .option("password", CLICKHOUSE_PROPS["password"])
    .option("driver", CLICKHOUSE_PROPS["driver"])
    .load()
    .first()
)

num_partizioni = 4
atleta_min = bounds["min_athlete_id"]
atleta_max = bounds["max_athlete_id"]

if atleta_min == 0 and atleta_max == 0:
    spark.stop()
    raise RuntimeError("Nessun allenamento disponibile per l'analisi.")

reader = (
    spark.read.format("jdbc")
    .option("url", CLICKHOUSE_URL)
    .option("dbtable", "allenamenti")
    .option("user", CLICKHOUSE_PROPS["user"])
    .option("password", CLICKHOUSE_PROPS["password"])
    .option("driver", CLICKHOUSE_PROPS["driver"])
)

if atleta_min == atleta_max:
    df = reader.load()
else:
    df = (
        reader
        .option("partitionColumn", "athlete_id")
        .option("lowerBound", atleta_min)
        .option("upperBound", atleta_max)
        .option("numPartitions", num_partizioni)
        .load()
    )

print("DataFrame caricato con successo")
df.show(5)

For each athlete, we want to select the exercise with the highest weight and repetitions in the same workout session.

In [ ]:
df_ordinato = df.filter(col("peso_allenamento").isNotNull())
df_ordinato.show(5)

window_spec = Window.partitionBy("athlete_id", "allenamento_id", "nome_esercizio") \
    .orderBy(F.col("peso_allenamento").desc(),F.col("ripetizioni_allenamento").desc())


df_ordinato = df_ordinato.withColumn("row_number", row_number().over(window_spec)) \
    .filter(col("row_number") == 1) \
    .drop("row_number")

df_ordinato.show(5)

Compute the theoretical maximum weight lifted for each exercise by each athlete.

In [ ]:
df_organizzato = df_ordinato.withColumn("massimale_teorico", F.col("peso_allenamento") * (1 + F.col("ripetizioni_allenamento") / 30))

df_organizzato.show(5)

df_organizzato = df_organizzato.drop("ripetizioni_allenamento","serie_allenamento","recupero_allenamento")

df_organizzato.show(5)

Select only the exercise done by a majority of athletes.

In [ ]:
SOGLIA_COPERTURA = 0.80
MIN_ATLETI_PER_ESERCIZIO = 30
MIN_SESSIONI_PER_ATLETA_ESERCIZIO = 3

atleti_validi = (
    df_organizzato
    .groupBy("athlete_id", "nome_esercizio")
    .agg(F.countDistinct("allenamento_id").alias("numero_sessioni"))
    .filter(F.col("numero_sessioni") >= MIN_SESSIONI_PER_ATLETA_ESERCIZIO)
)

numero_atleti = atleti_validi.select("athlete_id").distinct().count()

esercizi_ammessi = (
    atleti_validi
    .groupBy("nome_esercizio")
    .agg(F.countDistinct("athlete_id").alias("numero_atleti"))
    .filter(
        (F.col("numero_atleti") >= MIN_ATLETI_PER_ESERCIZIO)
        & (F.col("numero_atleti") >= numero_atleti * SOGLIA_COPERTURA)
    )
    .select("nome_esercizio")
)

df_organizzato = df_organizzato.join(
    esercizi_ammessi,
    on="nome_esercizio",
    how="inner",
)

df_organizzato.show(5)

Compute the variance of the theoretical maximum weight lifted for each exercise by each athlete.

In [ ]:
df_a = df_organizzato.alias("a")
df_b = df_organizzato.alias("b")

df_joined_12 = df_a.join(df_b,
                        (col("a.athlete_id") == col("b.athlete_id"))
                        & (col("b.data_allenamento") <= col("a.data_allenamento")) 
                        & (col("a.nome_esercizio") == col("b.nome_esercizio"))
                        & (col("b.data_allenamento") >= F.add_months(col("a.data_allenamento"), -12)),
                        "inner")

df_varianze_mobili = df_joined_12.groupBy("a.athlete_id", "a.allenamento_id", "a.nome_esercizio","a.data_allenamento","a.massimale_teorico") \
    .agg(F.variance("b.massimale_teorico").alias("varianza_12_mesi"),
            F.variance(F.when(col("b.data_allenamento") >= F.add_months(col("a.data_allenamento"), -6),
                            col("b.massimale_teorico"))).alias("varianza_6_mesi"),
            F.variance(F.when(col("b.data_allenamento") >= F.add_months(col("a.data_allenamento"), -3),
                            col("b.massimale_teorico"))).alias("varianza_3_mesi"),
            F.variance(F.when(col("b.data_allenamento") >= F.add_months(col("a.data_allenamento"), -1),
                            col("b.massimale_teorico"))).alias("varianza_1_mese"))


df_varianze_mobili.show(5)

Trasform the tables to prepare them for correlation analysis.

In [ ]:
df_preprocessing = df_varianze_mobili.select(
    "athlete_id",
    "allenamento_id",
    "nome_esercizio",
    "data_allenamento",
    "massimale_teorico",
    F.expr("stack(4, 'varianza_12_mesi', varianza_12_mesi," \
           " 'varianza_6_mesi', varianza_6_mesi, 'varianza_3_mesi', varianza_3_mesi, 'varianza_1_mese', varianza_1_mese) as (periodo, valore)")
)
df_preprocessing = df_preprocessing.withColumn("features_name", F.concat_ws("_", F.col("nome_esercizio"), F.col("periodo")))

df_preprocessing = df_preprocessing.groupBy(
    "athlete_id",
    "data_allenamento",
).pivot("features_name") \
    .agg(F.first("valore"))

df_preprocessing.show(5)

Compute the correlation matrix for the preprocessed exercise variance data.

In [ ]:
feature_columns = [
    column_name
    for column_name in df_preprocessing.columns
    if column_name not in ["athlete_id", "data_allenamento"]
]

MIN_OSSERVAZIONI_COPPIA = 30

if len(feature_columns) < 2:
    spark.stop()
    raise RuntimeError("Servono almeno due feature per calcolare la correlazione.")

pair_definitions = list(combinations(feature_columns, 2))
aggregate_expressions = []

for pair_index, (left_feature, right_feature) in enumerate(pair_definitions):
    valid_pair = (
        F.col(left_feature).isNotNull() & F.col(right_feature).isNotNull()
    )

    aggregate_expressions.extend([
        F.sum(F.when(valid_pair, 1).otherwise(0)).alias(f"pair_{pair_index}_count"),
        F.corr(
            F.when(valid_pair, F.col(left_feature)),
            F.when(valid_pair, F.col(right_feature)),
        ).alias(f"pair_{pair_index}_corr"),
    ])

pair_results = df_preprocessing.agg(*aggregate_expressions).first()

print("Correlazioni Pearson per coppie di esercizi:")
for pair_index, (left_feature, right_feature) in enumerate(pair_definitions):
    observations = pair_results[f"pair_{pair_index}_count"]
    correlation = pair_results[f"pair_{pair_index}_corr"]

    if observations is None or observations < MIN_OSSERVAZIONI_COPPIA:
        print(f"{left_feature} - {right_feature}: Not enough observations (only {observations or 0})")
        continue

    if correlation is None:
        print(f"{left_feature} - {right_feature}: Correlation could not be computed")
        continue

    print(f"{left_feature} - {right_feature}: Correlazione = {correlation}, Osservazioni = {observations}")

## Heatmap delle correlazioni

La matrice mostra solo le correlazioni con almeno il numero minimo di osservazioni condivise; le celle grigie non hanno dati sufficienti o una correlazione calcolabile.

In [ ]:
numero_feature = len(feature_columns)
correlation_matrix = np.full((numero_feature, numero_feature), np.nan, dtype=float)
observation_matrix = np.zeros((numero_feature, numero_feature), dtype=int)
np.fill_diagonal(correlation_matrix, 1.0)

feature_index = {
    feature_name: index
    for index, feature_name in enumerate(feature_columns)
}

for pair_index, (left_feature, right_feature) in enumerate(pair_definitions):
    observations = pair_results[f"pair_{pair_index}_count"] or 0
    correlation = pair_results[f"pair_{pair_index}_corr"]
    left_index = feature_index[left_feature]
    right_index = feature_index[right_feature]

    observation_matrix[left_index, right_index] = observations
    observation_matrix[right_index, left_index] = observations

    if observations >= MIN_OSSERVAZIONI_COPPIA and correlation is not None:
        correlation_matrix[left_index, right_index] = correlation
        correlation_matrix[right_index, left_index] = correlation

fig, axis = plt.subplots(
    figsize=(max(9, numero_feature * 0.75), max(7, numero_feature * 0.65)),
    dpi=120,
)

masked_matrix = np.ma.masked_invalid(correlation_matrix)
color_map = plt.get_cmap("RdBu_r").copy()
color_map.set_bad(color="#d9d9d9")
image = axis.imshow(masked_matrix, cmap=color_map, vmin=-1, vmax=1)

axis.set_xticks(range(numero_feature))
axis.set_yticks(range(numero_feature))
axis.set_xticklabels(feature_columns, rotation=90, fontsize=8)
axis.set_yticklabels(feature_columns, fontsize=8)
axis.set_title("Correlazioni Pearson delle varianze di massimale")

if numero_feature <= 15:
    for row_index in range(numero_feature):
        for column_index in range(numero_feature):
            correlation = correlation_matrix[row_index, column_index]
            if not np.isnan(correlation):
                axis.text(
                    column_index,
                    row_index,
                    f"{correlation:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

colorbar = fig.colorbar(image, ax=axis)
colorbar.set_label("Correlazione Pearson r")
fig.tight_layout()
plt.show()

print("Matrice delle osservazioni condivise:")
print(observation_matrix)

## Benchmark della correlazione

Per confrontare configurazioni diverse, modifica `SPARK_EXECUTOR_INSTANCES`, riavvia la SparkSession e riesegui questa cella. Il benchmark misura soltanto l'aggregazione pairwise: pipeline e dataset restano invariati.

In [ ]:
inizio_benchmark = time.perf_counter()

# Stessa aggregazione pairwise della cella precedente: il carico confrontato resta identico.
risultato_benchmark = df_preprocessing.agg(*aggregate_expressions).first()
durata_benchmark = time.perf_counter() - inizio_benchmark

executor_memory_status = spark.sparkContext._jsc.sc().getExecutorMemoryStatus()
executor_effettivi = max(0, executor_memory_status.size() - 1)

print("Benchmark correlazione pairwise")
print(f"executor richiesti: {NUM_EXECUTORS}")
print(f"executor effettivi: {executor_effettivi}")
print(f"feature:            {len(feature_columns)}")
print(f"coppie:             {len(pair_definitions)}")
print(f"tempo benchmark:    {durata_benchmark:.2f} s")

# Mantiene il risultato materializzato per evitare che il benchmark venga ottimizzato via.
assert risultato_benchmark is not None

Close the Spark session.

In [ ]:
spark.stop()
print("SparkSession closed.")